In [ ]:
# Week 6 Project - Deep Learning

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

# Reproducibility: fixed seeds so results can be reproduced
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
(x_train, _), (x_test, _) = mnist.load_data()

In [ ]:
print("Original training shape:", x_train.shape)  # (60000, 28, 28)
print("Original test shape:    ", x_test.shape)   # (10000, 28, 28)

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

In [ ]:
x_train = np.expand_dims(x_train, axis=-1)
x_test  = np.expand_dims(x_test,  axis=-1)

print("Processed training shape:", x_train.shape)  # (60000, 28, 28, 1)
print("Pixel value range:", x_train.min(), "to", x_train.max())

In [ ]:
noise_factor = 0.5

x_train_noisy = x_train + noise_factor * np.random.normal(
    loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy = x_test + noise_factor * np.random.normal(
    loc=0.0, scale=1.0, size=x_test.shape)

In [ ]:
x_train_noisy = np.clip(x_train_noisy, 0.0, 1.0)
x_test_noisy  = np.clip(x_test_noisy,  0.0, 1.0)

print("Noisy training shape:", x_train_noisy.shape)

In [ ]:
n = 8
plt.figure(figsize=(16, 4))
for i in range(n):
    # Clean
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title("Clean"); plt.axis("off")
    # Noisy
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(x_train_noisy[i].squeeze(), cmap="gray")
    plt.title("Noisy"); plt.axis("off")
plt.suptitle("Clean (top) vs Noisy (bottom) - Training Inputs")
plt.show()

In [ ]:
def build_denoising_autoencoder():
    input_img = layers.Input(shape=(28, 28, 1), name="noisy_input")

    # ----- ENCODER -----
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(input_img)
    x = layers.MaxPooling2D((2, 2), padding="same")(x)          # 28x28 -> 14x14
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    encoded = layers.MaxPooling2D((2, 2), padding="same")(x)    # 14x14 -> 7x7
    

    # ----- DECODER -----
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(encoded)
    x = layers.UpSampling2D((2, 2))(x)                          # 7x7 -> 14x14
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    x = layers.UpSampling2D((2, 2))(x)                          # 14x14 -> 28x28
    decoded = layers.Conv2D(1, (3, 3), activation="sigmoid",
                            padding="same", name="clean_output")(x)
    

    autoencoder = models.Model(input_img, decoded, name="denoising_autoencoder")
    return autoencoder

autoencoder = build_denoising_autoencoder()

In [ ]:
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")

autoencoder.summary()

In [ ]:
history = autoencoder.fit(
    x_train_noisy, x_train,          # noisy -> clean
    epochs=20,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_noisy, x_test),
    verbose=1
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.grid(True)
plt.show()

In [ ]:
denoised_imgs = autoencoder.predict(x_test_noisy)
print("Denoised output shape:", denoised_imgs.shape)

In [ ]:
n = 10
plt.figure(figsize=(20, 6))
for i in range(n):
    # Row 1: Original (clean) test images
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    if i == 0: ax.set_ylabel("Original", fontsize=12)
    plt.xticks([]); plt.yticks([])

    # Row 2: Noisy input images
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].squeeze(), cmap="gray")
    if i == 0: ax.set_ylabel("Noisy", fontsize=12)
    plt.xticks([]); plt.yticks([])

    # Row 3: Denoised (reconstructed) images
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(denoised_imgs[i].squeeze(), cmap="gray")
    if i == 0: ax.set_ylabel("Denoised", fontsize=12)
    plt.xticks([]); plt.yticks([])

plt.suptitle("Original vs Noisy vs Denoised (Test Set)", fontsize=16)
plt.show()

In [ ]:
def psnr(clean, other):
    return tf.image.psnr(clean, other, max_val=1.0).numpy().mean()

psnr_noisy    = psnr(x_test, x_test_noisy)      # before denoising
psnr_denoised = psnr(x_test, denoised_imgs)     # after denoising

print(f"Average PSNR (Noisy vs Clean):    {psnr_noisy:.2f} dB")
print(f"Average PSNR (Denoised vs Clean): {psnr_denoised:.2f} dB")
print(f"Improvement: +{psnr_denoised - psnr_noisy:.2f} dB")